# Hidden State Extraction — All Four Conditions

Extracts `h_inst` and `h_post_inst` at the Zhao et al. token positions under all four extraction conditions used in the study. Merged from the former nb03 (trajectory + single-turn baseline) and nb07 (no-context + compressed).

| # | Condition | Input | Granularity | Output directory |
|---|-----------|-------|-------------|------------------|
| 1 | Full-context | turns 1..k, prior assistant replies included | per (conv, k) | `trajectories/{fw}_{split}/` |
| 2 | No-context | system + user_k only | per (conv, k) | `nocontext/{fw}_{split}/` |
| 3 | Compressed | full conversation flattened to one user msg | per conv | `compressed/{fw}_{split}/` |
| 4 | Single-turn baseline | raw JBB goal, no attack framing | per goal | `single_turn/{harmful,benign}/` |

```
... [user content] <|eot_id|>
         ^              ^
      t_inst       t_post_inst
```

All conditions save `(N, n_layers, hidden_dim)` float16 arrays + `metadata.parquet`. Sections are independently runnable — each condition has resume support, so re-running skips rows already present on disk.

Shared logic lives in `src/extraction/runner.py`.

---

**WildJailbreak variant.** This notebook reads conversations from
`data/llama/conversations_wj/` and writes representations to
`data/llama/representations_wj/`, leaving the JBB outputs untouched.
The single-turn baseline draws from the 50 + 50 OOD-selected WildJailbreak
goals (`data/selection/wildjailbreak_selected_v1.parquet`) so all four
extraction conditions share the same source goals.


## 1. Setup

In [1]:
import os
# Set visible GPUs before importing torch.
GPU_IDS = [4]                     # physical GPU IDs
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in GPU_IDS)
LOGICAL_GPU_IDS = list(range(len(GPU_IDS)))

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.extraction.runner import (
    get_accepted_turns,
    build_fullcontext_messages,
    build_nocontext_messages,
    build_compressed_messages,
    get_positions,
    iter_trajectory_units,
    iter_nocontext_units,
    iter_compressed_units,
    make_base_meta_fn,
    run_condition,
    run_single_turn_baseline,
)

In [11]:
# ── Per-run config ────────────────────────────────────────────────────────────
MODEL       = "llama"                   # this notebook's model folder (notebooks/llama/ → data/llama/)
FOLDER_NAME = "actorattack_benign"       # one of: {fw}_{harmful|benign}

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
DTYPE    = torch.bfloat16
USER_END_TOKEN = "<|eot_id|>"           # Llama-3.x; needs per-model values for Qwen/Mistral

# Llama-3.1-8B has 32 transformer blocks. Extract at 8 proportional layers
# (np.linspace(0, 31, 8) + 1) — same sweep convention used across models and
# by the shared analysis notebooks via layer_indices.json.
N_TRANSFORMER_LAYERS_TOTAL = 32
LAYER_INDICES = (np.linspace(0, N_TRANSFORMER_LAYERS_TOTAL - 1, 8, dtype=int) + 1).tolist()
N_LAYERS = len(LAYER_INDICES)

TRAJ_REPS = None    # None → all attempts; set e.g. {1,2,3,4,5} to subsample for dev

# ── Derived paths ─────────────────────────────────────────────────────────────
CONV_DIR  = repo_root / "data" / MODEL / "conversations_wj" / FOLDER_NAME
REPR_ROOT = repo_root / "data" / MODEL / "representations_wj"

FRAMEWORK, SPLIT = FOLDER_NAME.split("_", 1)

assert CONV_DIR.exists(), f"Conversation folder not found: {CONV_DIR}"
files = sorted(CONV_DIR.glob("*.json"))
if TRAJ_REPS is not None:
    files = [f for f in files
             if json.loads(f.read_text()).get("attempt", 1) in TRAJ_REPS]

base_meta_fn = make_base_meta_fn(FRAMEWORK, SPLIT)

print(f"Folder:    {FOLDER_NAME}")
print(f"Framework: {FRAMEWORK}    Split: {SPLIT}")
print(f"Files:     {len(files)}" + (f'  (filtered by attempts {sorted(TRAJ_REPS)})' if TRAJ_REPS else ''))
print(f"Model:     {MODEL_ID}")
print(f"GPUs:      {GPU_IDS} → logical {LOGICAL_GPU_IDS}")
print(f"Layers:    {LAYER_INDICES}  ({N_LAYERS} of {N_TRANSFORMER_LAYERS_TOTAL})")

Folder:    actorattack_benign
Framework: actorattack    Split: benign
Files:     1000
Model:     meta-llama/Llama-3.1-8B-Instruct
GPUs:      [4] → logical [0]
Layers:    [1, 5, 9, 14, 18, 23, 27, 32]  (8 of 32)


## 2. Sanity checks

Run before any large extraction.

1. Token positions — `t_inst` should be a content token, `t_post_inst` should be `<|eot_id|>` (or the model's user-end token)
2. Prefix construction — full-context at k produces 1 system + k user + (k-1) assistant messages
3. No-context user content matches full-context user content at each k
4. Accepted-turn count matches `executed_turns` in the conversation JSON (rolled-back filtering)

In [12]:
tok_check = AutoTokenizer.from_pretrained(MODEL_ID)

print("=" * 60)
print("CHECK 1 — Token positions (full-context, final turn)")
print("=" * 60)

for fpath in files[:5]:
    conv = json.loads(fpath.read_text())
    accepted = get_accepted_turns(conv)
    if not accepted:
        print(f"  {fpath.name}: SKIPPED (no accepted turns)")
        continue

    messages = build_fullcontext_messages(conv, len(accepted))
    input_ids = tok_check.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=False
    )
    t_inst, t_post = get_positions(tok_check, input_ids, USER_END_TOKEN)

    tok_inst = tok_check.decode([input_ids[0][t_inst].item()])
    tok_post = tok_check.decode([input_ids[0][t_post].item()])
    context  = tok_check.decode(input_ids[0][t_inst - 3 : t_post + 2].tolist())

    ok_inst = bool(tok_inst.strip())
    ok_post = tok_post == USER_END_TOKEN

    print(f"  {fpath.name}  turns={len(accepted)}")
    print(f"    t_inst [{t_inst}]      = {repr(tok_inst):20s}  {'OK' if ok_inst else 'WARN: whitespace'}")
    print(f"    t_post_inst [{t_post}] = {repr(tok_post):20s}  {'OK' if ok_post else f'WARN: not {USER_END_TOKEN}'}")
    print(f"    context: {repr(context)}")
    print()

# Also verify no-context and compressed on one conversation
conv0 = json.loads(files[0].read_text())

print("-- No-context (k=1) --")
nc_msgs = build_nocontext_messages(conv0, 1)
nc_ids  = tok_check.apply_chat_template(nc_msgs, return_tensors="pt", add_generation_prompt=False)
t_i, t_p = get_positions(tok_check, nc_ids, USER_END_TOKEN)
print(f"  t_inst={repr(tok_check.decode([nc_ids[0][t_i].item()]))}  "
      f"t_post={repr(tok_check.decode([nc_ids[0][t_p].item()]))}")

print("-- Compressed --")
comp_msgs = build_compressed_messages(conv0)
comp_ids  = tok_check.apply_chat_template(comp_msgs, return_tensors="pt", add_generation_prompt=False)
t_i, t_p = get_positions(tok_check, comp_ids, USER_END_TOKEN)
print(f"  t_inst={repr(tok_check.decode([comp_ids[0][t_i].item()]))}  "
      f"t_post={repr(tok_check.decode([comp_ids[0][t_p].item()]))}")

CHECK 1 — Token positions (full-context, final turn)
  actorattack_llama3_8b_pair000_attempt1.json: SKIPPED (no accepted turns)
  actorattack_llama3_8b_pair000_attempt10.json  turns=6
    t_inst [3034]      = '.'                   OK
    t_post_inst [3035] = '<|eot_id|>'          OK
    context: ' the above content.<|eot_id|>'

  actorattack_llama3_8b_pair000_attempt11.json  turns=6
    t_inst [2723]      = '.'                   OK
    t_post_inst [2724] = '<|eot_id|>'          OK
    context: ' the above content.<|eot_id|>'

  actorattack_llama3_8b_pair000_attempt12.json  turns=6
    t_inst [3071]      = '.'                   OK
    t_post_inst [3072] = '<|eot_id|>'          OK
    context: ' the above content.<|eot_id|>'

  actorattack_llama3_8b_pair000_attempt13.json  turns=6
    t_inst [3367]      = '.'                   OK
    t_post_inst [3368] = '<|eot_id|>'          OK
    context: ' the above content.<|eot_id|>'

-- No-context (k=1) --


UndefinedError: None has no element 0

In [6]:
conv = json.loads(files[0].read_text())
accepted = get_accepted_turns(conv)

print("=" * 60)
print("CHECK 2 — Prefix construction (1 system + k user + (k-1) assistant)")
print("=" * 60)
for k in [1, 2, len(accepted)]:
    msgs = build_fullcontext_messages(conv, k)
    expected_len = 1 + k + (k - 1)
    status = "OK" if len(msgs) == expected_len else f"MISMATCH (expected {expected_len})"
    print(f"  k={k}: {len(msgs)} messages  {status}")

print()
print("=" * 60)
print("CHECK 3 — No-context user content matches full-context user content")
print("=" * 60)
for k in [1, 2, len(accepted)]:
    full_msgs = build_fullcontext_messages(conv, k)
    nc_msgs   = build_nocontext_messages(conv, k)
    match = full_msgs[-1]["content"] == nc_msgs[-1]["content"]
    print(f"  k={k}: full={len(full_msgs)} msgs, nocontext={len(nc_msgs)} msgs  match: {match}")
    assert match, f"Content mismatch at k={k}!"

print()
print("=" * 60)
print("CHECK 4 — Rolled-back turn filtering")
print("=" * 60)
for fpath in files[:10]:
    conv = json.loads(fpath.read_text())
    n_accepted = len(get_accepted_turns(conv))
    expected   = conv.get("executed_turns", n_accepted)
    status = "OK" if n_accepted == expected else f"MISMATCH (executed_turns={expected})"
    print(f"  {fpath.name}: accepted={n_accepted}  {status}")

CHECK 2 — Prefix construction (1 system + k user + (k-1) assistant)
  k=1: 2 messages  OK
  k=2: 4 messages  OK
  k=6: 12 messages  OK

CHECK 3 — No-context user content matches full-context user content
  k=1: full=2 msgs, nocontext=2 msgs  match: True
  k=2: full=4 msgs, nocontext=2 msgs  match: True
  k=6: full=12 msgs, nocontext=2 msgs  match: True

CHECK 4 — Rolled-back turn filtering
  actorattack_llama3_8b_pair000_attempt1.json: accepted=6  OK
  actorattack_llama3_8b_pair000_attempt10.json: accepted=6  OK
  actorattack_llama3_8b_pair000_attempt11.json: accepted=6  OK
  actorattack_llama3_8b_pair000_attempt12.json: accepted=6  OK
  actorattack_llama3_8b_pair000_attempt13.json: accepted=6  OK
  actorattack_llama3_8b_pair000_attempt14.json: accepted=6  OK
  actorattack_llama3_8b_pair000_attempt15.json: accepted=6  OK
  actorattack_llama3_8b_pair000_attempt16.json: accepted=6  OK
  actorattack_llama3_8b_pair000_attempt17.json: accepted=6  OK
  actorattack_llama3_8b_pair000_attempt18

## 3. Condition 4 — Single-turn baseline

Raw JBB goals (100 harmful + 100 benign) with no attack framing. One extraction per goal. Produces the anchor used as `v_harmful(ST)` throughout nb08/09/10.

Run **once per model** — not per-framework, not per-split. If the output folders already exist, this cell is a no-op.

In [7]:
SHARED_SYSTEM_PROMPT = (
    "You are a helpful, respectful and honest assistant. Always answer as helpfully "
    "as possible, while being safe.\n\nYour answers should not include any harmful, "
    "unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure "
    "that your responses are socially unbiased and positive in nature.\n\nIf a "
    "question does not make any sense, or is not factually coherent, explain why "
    "instead of answering something not correct. If you don't know the answer to a "
    "question, please don't share false information."
)

sel_path = repo_root / "data" / "selection" / "wildjailbreak_selected_v1.parquet"
sel = pd.read_parquet(sel_path)

def _to_goals_df(df):
    # Adapter: produce a frame with the columns run_single_turn_baseline reads.
    df = df.sort_values("max_cos_to_jbb").reset_index(drop=True)
    return pd.DataFrame({
        "Index":    df.index.astype(int),
        "Goal":     df["vanilla"].values,
        "Behavior": [f"wj_{int(i):05d}" for i in df["wj_idx"].values],
        "Category": "wildjailbreak",
    })

wj_goals = {
    "harmful": _to_goals_df(sel[sel["goal_type"] == "harmful"]),
    "benign":  _to_goals_df(sel[sel["goal_type"] == "benign"]),
}

for goal_type in ["harmful", "benign"]:
    save_dir = REPR_ROOT / "single_turn" / goal_type
    if (save_dir / "metadata.parquet").exists():
        print(f"skip {save_dir} (already extracted)")
        continue
    run_single_turn_baseline(
        goal_type=goal_type,
        goals_df=wj_goals[goal_type],
        system_prompt=SHARED_SYSTEM_PROMPT,
        save_dir=save_dir,
        model_id=MODEL_ID,
        logical_gpu_ids=LOGICAL_GPU_IDS,
        layer_indices=LAYER_INDICES,
        dtype=DTYPE,
        user_end_token=USER_END_TOKEN,
        n_transformer_layers_total=N_TRANSFORMER_LAYERS_TOTAL,
    )


skip /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/llama/representations_wj/single_turn/harmful (already extracted)
skip /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/llama/representations_wj/single_turn/benign (already extracted)


## 4. Condition 1 — Full-context trajectory

For each conversation of length n, runs n forward passes at prefix depths k = 1..n. The hidden state at each k reflects both user_k content and accumulated prior turns.

Resume key: `(pair_id, attempt, turn_k)`.

In [8]:
run_condition(
    files=files,
    iter_units=iter_trajectory_units,
    base_meta_fn=base_meta_fn,
    save_dir=REPR_ROOT / "trajectories" / FOLDER_NAME,
    model_id=MODEL_ID,
    logical_gpu_ids=LOGICAL_GPU_IDS,
    layer_indices=LAYER_INDICES,
    dtype=DTYPE,
    resume_key_cols=("pair_id", "attempt", "turn_k"),
    desc=f"Trajectory {FOLDER_NAME}",
    user_end_token=USER_END_TOKEN,
    n_transformer_layers_total=N_TRANSFORMER_LAYERS_TOTAL,
)

Resuming: 5982 entries already present in actorattack_harmful
Total units: 5982   Remaining: 0


Trajectory actorattack_harmful: 0it [00:00, ?it/s]

GPU 0: loading meta-llama/Llama-3.1-8B-Instruct ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GPU 0: ready, 1000 files
  Saved → /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/llama/representations_wj/trajectories/actorattack_harmful
  h_inst:      (5982, 8, 4096)  (0.39 GB)
  h_post_inst: (5982, 8, 4096)  (0.39 GB)
  rows:        5982


## 5. Condition 2 — No-context

Same per-turn indexing as trajectory, but each forward pass contains only `[system]` + `user_k` — no prior turns. The difference with full-context at the same (conv, k) isolates the pure effect of context accumulation.

Resume key: `(pair_id, attempt, turn_k)`.

In [9]:
run_condition(
    files=files,
    iter_units=iter_nocontext_units,
    base_meta_fn=base_meta_fn,
    save_dir=REPR_ROOT / "nocontext" / FOLDER_NAME,
    model_id=MODEL_ID,
    logical_gpu_ids=LOGICAL_GPU_IDS,
    layer_indices=LAYER_INDICES,
    dtype=DTYPE,
    resume_key_cols=("pair_id", "attempt", "turn_k"),
    desc=f"No-context {FOLDER_NAME}",
    user_end_token=USER_END_TOKEN,
    n_transformer_layers_total=N_TRANSFORMER_LAYERS_TOTAL,
)

Resuming: 5982 entries already present in actorattack_harmful
Total units: 5982   Remaining: 0


No-context actorattack_harmful: 0it [00:00, ?it/s]

GPU 0: loading meta-llama/Llama-3.1-8B-Instruct ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GPU 0: ready, 1000 files
  Saved → /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/llama/representations_wj/nocontext/actorattack_harmful
  h_inst:      (5982, 8, 4096)  (0.39 GB)
  h_post_inst: (5982, 8, 4096)  (0.39 GB)
  rows:        5982


## 6. Condition 3 — Compressed

Full conversation concatenated into one user message with plain-text `User:` / `Assistant:` prefixes (no `<|eot_id|>` between turns). One extraction per conversation.

Resume key: `(pair_id, attempt)`.

In [10]:
run_condition(
    files=files,
    iter_units=iter_compressed_units,
    base_meta_fn=base_meta_fn,
    save_dir=REPR_ROOT / "compressed" / FOLDER_NAME,
    model_id=MODEL_ID,
    logical_gpu_ids=LOGICAL_GPU_IDS,
    layer_indices=LAYER_INDICES,
    dtype=DTYPE,
    resume_key_cols=("pair_id", "attempt"),
    desc=f"Compressed {FOLDER_NAME}",
    user_end_token=USER_END_TOKEN,
    n_transformer_layers_total=N_TRANSFORMER_LAYERS_TOTAL,
)

Resuming: 997 entries already present in actorattack_harmful
Total units: 997   Remaining: 0


Compressed actorattack_harmful: 0it [00:00, ?it/s]

GPU 0: loading meta-llama/Llama-3.1-8B-Instruct ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GPU 0: ready, 1000 files
  Saved → /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/llama/representations_wj/compressed/actorattack_harmful
  h_inst:      (997, 8, 4096)  (0.07 GB)
  h_post_inst: (997, 8, 4096)  (0.07 GB)
  rows:        997


## 7. Cross-condition verification

**Check 5:** At k=1, no-context and full-context receive identical inputs (no prior turns exist). Cosine similarity should be ≈ 1.0.

**Check 6:** At k > 1, the two conditions should progressively diverge — the gap is what the displacement analysis quantifies.

In [ ]:
traj_dir = REPR_ROOT / "trajectories" / FOLDER_NAME
nc_dir   = REPR_ROOT / "nocontext"    / FOLDER_NAME

if not (traj_dir.exists() and nc_dir.exists()):
    print(f"Missing trajectory or no-context output for {FOLDER_NAME} — run sections 4 & 5 first.")
else:
    traj_meta = pd.read_parquet(traj_dir / "metadata.parquet")
    traj_h    = np.load(str(traj_dir / "h_inst.npy"), mmap_mode="r")
    nc_meta   = pd.read_parquet(nc_dir / "metadata.parquet")
    nc_h      = np.load(str(nc_dir / "h_inst.npy"), mmap_mode="r")

    # Saved-array positions (0..7) — labels come from LAYER_INDICES we just extracted.
    n_saved = len(LAYER_INDICES)
    CHECK_POSITIONS = [n_saved // 4, n_saved // 2, n_saved - 1]
    CHECK_LABELS    = [f"L{LAYER_INDICES[p]}" for p in CHECK_POSITIONS]
    MID_POSITION    = n_saved // 2
    MID_LABEL       = f"L{LAYER_INDICES[MID_POSITION]}"

    def cosine(a, b):
        a, b = a.astype(np.float32), b.astype(np.float32)
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)

    print("=" * 60)
    print("CHECK 5 — k=1 cosine (expected ~1.0)")
    print("=" * 60)
    traj_k1 = traj_meta[traj_meta["turn_k"] == 1].head(10)
    for _, row in traj_k1.iterrows():
        match = nc_meta[
            (nc_meta["pair_id"] == row["pair_id"])
            & (nc_meta["attempt"] == row["attempt"])
            & (nc_meta["turn_k"] == 1)
        ]
        if match.empty:
            continue
        sims = [cosine(traj_h[row.name, p, :], nc_h[match.index[0], p, :])
                for p in CHECK_POSITIONS]
        print(f"  pair={row['pair_id']:3d} att={row['attempt']}  "
              + "  ".join(f"{ll}={s:.6f}" for ll, s in zip(CHECK_LABELS, sims)))

    print()
    print("=" * 60)
    print(f"CHECK 6 — divergence at k>1 ({MID_LABEL})")
    print("=" * 60)
    for k_check in [1, 3, 5, 8]:
        traj_rows = traj_meta[traj_meta["turn_k"] == k_check].head(30)
        sims = []
        for _, row in traj_rows.iterrows():
            match = nc_meta[
                (nc_meta["pair_id"] == row["pair_id"])
                & (nc_meta["attempt"] == row["attempt"])
                & (nc_meta["turn_k"] == k_check)
            ]
            if match.empty:
                continue
            sims.append(cosine(traj_h[row.name, MID_POSITION, :],
                               nc_h[match.index[0], MID_POSITION, :]))
        if sims:
            print(f"  k={k_check:2d}: mean cos {MID_LABEL} = {np.mean(sims):.4f} ± {np.std(sims):.4f}  (n={len(sims)})")
    print("\nExpected: k=1 ~ 1.0; k>1 progressively < 1.0")

CHECK 5 — k=1 cosine (expected ~1.0)
  pair=  0 att=1  L9=1.000000  L18=1.000000  L32=1.000000
  pair=  0 att=10  L9=1.000000  L18=1.000000  L32=1.000000
  pair=  0 att=11  L9=1.000000  L18=1.000000  L32=1.000000
  pair=  0 att=12  L9=1.000000  L18=1.000000  L32=1.000000
  pair=  0 att=13  L9=1.000000  L18=1.000000  L32=1.000000
  pair=  0 att=14  L9=1.000000  L18=1.000000  L32=1.000000
  pair=  0 att=15  L9=1.000000  L18=1.000000  L32=1.000000
  pair=  0 att=16  L9=1.000000  L18=1.000000  L32=1.000000
  pair=  0 att=17  L9=1.000000  L18=1.000000  L32=1.000000
  pair=  0 att=18  L9=1.000000  L18=1.000000  L32=1.000000

CHECK 6 — divergence at k>1 (L18)
  k= 1: mean cos L18 = 1.0000 ± 0.0000  (n=30)
  k= 3: mean cos L18 = 0.8940 ± 0.0337  (n=30)
  k= 5: mean cos L18 = 0.8579 ± 0.0645  (n=30)
  k= 8: mean cos L18 = 0.8255 ± 0.0714  (n=30)

Expected: k=1 ~ 1.0; k>1 progressively < 1.0


: 